In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
checkpoint_path = 'last_checkpoint_unmasking.pt'

model = TinyStoriesLM.load(checkpoint_path, device=device)

# We have to freeze all the model
for param in model.parameters():
    param.requires_grad = False

# We set our model i eval mode -> deactivate dropout  so the vectors are stable
# dropout turns off random neurons during training to avoid overfitting/memorizing 
model.eval() #same input = same output

activations = {}

def forward_hook(module, input, output): # hook function
    # 'output' - ouput from Transformer layer
    # module layer that was just executed
    # save in our dictionary
    # .detach() no gradient info / history - we just want numbers
    
    activations[module.name] = output.detach() # save

hooks = []
# register each block (TransformerBlock)
for i, block in enumerate(model.transformers):
    block.name = f"layer_{i}" 
    handle = block.register_forward_hook(forward_hook)
    hooks.append(handle)


for layer, act_vector in activations.items():
    print(f"{layer} shape: {act_vector.shape}")



#.remove IMPOOO

# when trainning (ignore_index=-100) impo
